<a href="https://colab.research.google.com/github/NirtonAfonso/tech-challenge-fase3-medflow-ai/blob/develop/notebooks/01_data_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Colab"/></a>

# 01 — Preparação de dados, anonimização e curadoria

**Tech Challenge Fase 3 — MedFlow AI**

Demonstra, de forma reprodutível, as três etapas obrigatórias de preparação de dados exigidas pelo
enunciado: **preprocessing**, **anonimização** e **curadoria**.

| | |
|---|---|
| Runtime | **CPU** (não precisa de GPU) |
| Como rodar | `Ambiente de execução` → `Executar tudo` |
| Saída no Drive | `MedFlowAI_Fase3/01_preprocessing/` |

Cada seção responde a uma pergunta explícita e termina com interpretação — não são gráficos soltos.

> ⚠️ Todos os dados são **sintéticos**. Nenhum dado real de paciente é utilizado.

In [ ]:
# @title ▶ Bootstrap — execute esta célula primeiro (Colab ou local)
#
# Prepara tudo do zero em um runtime Colab novo: monta o Google Drive, clona a
# branch `develop`, instala as dependências e cria a estrutura de saída.
# Rodando localmente, detecta o repositório e pula clone/Drive.

import os
import pathlib
import subprocess
import sys

REPO_URL = "https://github.com/NirtonAfonso/tech-challenge-fase3-medflow-ai.git"
REPO_BRANCH = "develop"
REPO_DIR = "tech-challenge-fase3-medflow-ai"
NOTEBOOK_ID = "01_preprocessing"
REQUIREMENTS = "requirements-colab.txt"

IN_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))


def _run(*args, **kwargs):
    return subprocess.run(list(args), check=kwargs.pop("check", True), **kwargs)


def _pip(*args):
    _run(sys.executable, "-m", "pip", *args)


def _tem_torch_cuda() -> bool:
    try:
        import torch

        return torch.cuda.is_available()
    except Exception:
        return False


# --- 1. Google Drive ---------------------------------------------------------
if IN_COLAB:
    try:
        from google.colab import drive

        drive.mount("/content/drive")
        print("Google Drive montado em /content/drive")
    except Exception as erro:
        print(f"ATENÇÃO: falha ao montar o Drive ({erro}).")
        print("Os resultados ficarão apenas em /content e serão PERDIDOS ao encerrar a sessão.")

# --- 2. Repositório ----------------------------------------------------------
def _raiz_local() -> pathlib.Path | None:
    atual = pathlib.Path.cwd()
    for candidato in [atual, *atual.parents]:
        if (candidato / "src" / "medflow_ai").exists():
            return candidato
    return None


raiz = _raiz_local()
if raiz is None:
    destino = pathlib.Path("/content" if IN_COLAB else ".") / REPO_DIR
    if destino.exists():
        _run("git", "-C", str(destino), "fetch", "--depth", "1", "origin", REPO_BRANCH)
        _run("git", "-C", str(destino), "checkout", REPO_BRANCH)
        _run("git", "-C", str(destino), "pull", "--ff-only", "origin", REPO_BRANCH)
    else:
        # Sempre com --branch explícita: nunca clonar a default implicitamente.
        _run("git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(destino))
    raiz = destino.resolve()

os.chdir(raiz)
if str(raiz / "src") not in sys.path:
    sys.path.insert(0, str(raiz / "src"))
print(f"Raiz do projeto: {raiz}")

# --- 3. Dependências ---------------------------------------------------------
_pip("install", "-q", "-U", "pip")
_pip("install", "-q", "-r", REQUIREMENTS)
_pip("install", "-q", "-e", ".")

# --- 4. Diagnóstico ----------------------------------------------------------
import platform

from medflow_ai.colab import ensure_structure, git_info, in_colab, write_run_metadata

_git = git_info(raiz)
print("\n" + "=" * 78)
print(f"Python        : {platform.python_version()}")
print(f"Ambiente      : {'Google Colab' if IN_COLAB else 'local'}")
print(f"Branch        : {_git['branch']}")
print(f"Commit        : {_git['commit']}")
print("=" * 78)

# --- 6. Estrutura de saída (Drive no Colab, artifacts/colab localmente) -------
PASTAS = ensure_structure(NOTEBOOK_ID)
print("\nEstrutura de saída:")
for _nome, _caminho in sorted(PASTAS.items()):
    print(f"  {_nome:22s} {_caminho}")

RUN_META = write_run_metadata(NOTEBOOK_ID)
print(f"\nMetadados da execução: {RUN_META}")


## 1. Qual é o corpus institucional e de onde ele vem?

15 documentos Markdown versionados em `data/synthetic/protocols/`: protocolos clínicos,
procedimentos internos, modelos de laudo e receita, e perguntas frequentes de médicos — exatamente as
categorias que o enunciado pede para o fine-tuning.

Manter o corpus em texto (e não em PDF binário) torna cada alteração rastreável pelo próprio `git`.

In [ ]:
from medflow_ai.data.corpus import load_corpus

documentos = load_corpus()
print(f"{len(documentos)} documentos | {sum(len(d.sections) for d in documentos)} seções\n")
for doc in documentos:
    print(f"{doc.doc_id:14s} v{doc.version:5s} {doc.doc_type:16s} {len(doc.sections):2d} seções  {doc.title[:52]}")

In [ ]:
# Origem declarada de cada documento: exigência de rastreabilidade.
for doc in documentos[:3]:
    print(f"{doc.doc_id}: {doc.metadata['origem']}\n")

**Interpretação.** Os 15 documentos cobrem 6 especialidades e 4 tipos documentais. Nenhum é
documento oficial: todos declaram explicitamente a origem sintética, o que evita atribuir ao
Ministério da Saúde um conteúdo que não é dele.

**Limitação.** Um hospital real teria centenas de protocolos; o corpus aqui é dimensionado para caber
no repositório e permitir avaliação reprodutível em CI.

## 2. A anonimização realmente remove identificadores? (antes / depois)

Evidência central do requisito de anonimização. O banco sintético é construído **aqui**, para que este
notebook não dependa de nenhum outro ter rodado antes.

In [ ]:
from medflow_ai.database.ingest import build_synthetic_database
from medflow_ai.database.repository import PatientRepository

contagens = build_synthetic_database(n_patients=40)
print("Banco sintético construído:", contagens)

repo = PatientRepository()
bruto = repo.raw_record("P-DEMO-0001")

print("\n=== ANTES (registro bruto, com identificadores diretos) ===")
for chave, valor in bruto.items():
    print(f"  {chave:12s}: {valor}")

In [ ]:
import json

from medflow_ai.data.anonymization import anonymize_record

anonimizado, relatorio_anon = anonymize_record(bruto)

print("=== DEPOIS (registro anonimizado) ===")
for chave, valor in anonimizado.items():
    print(f"  {chave:12s}: {valor}")

print("\n=== RELATÓRIO DE ANONIMIZAÇÃO ===")
print(json.dumps(relatorio_anon.to_dict(), ensure_ascii=False, indent=2))

In [ ]:
from medflow_ai.data.anonymization import anonymize_text

texto = (
    "Paciente: Maria da Silva Souza, CPF 123.456.789-00, CNS 700 5049 3417 8563, "
    "e-mail maria.souza@exemplo.com.br, telefone (11) 98765-4321, residente na "
    "Rua das Acácias, nº 120, CEP 01310-100. Data de nascimento: 12/03/1975. "
    "Prontuário nº 4457821. Atendida pelo Dr. Carlos Andrade, CRM/SP 123456. "
    "Diagnóstico: hipotireoidismo primário, TSH 8,4 mUI/L."
)
limpo, rel = anonymize_text(texto, redact_dates=True)
print("ANTES :", texto, "\n")
print("DEPOIS:", limpo, "\n")
print("Identificadores encontrados por tipo:", rel.counts_by_kind)

**Interpretação.** Oito classes de identificador direto são removidas; o conteúdo clínico
(`TSH 8,4 mUI/L`, `hipotireoidismo primário`) é integralmente preservado. A data de nascimento é
*transformada* em idade e faixa etária, e não simplesmente apagada — preservando utilidade clínica com
menor risco de reidentificação.

**Trade-off.** O reconhecimento de nome depende de rótulo explícito (`Paciente:`, `Dr.`). Um nome solto
no meio da frase não é capturado. A alternativa (NER treinado) traria falsos positivos sobre termos
clínicos e uma dependência pesada; a escolha privilegia previsibilidade e auditabilidade.

**Limitação.** Cidade/UF permanecem no texto (quase-identificador). Em produção, seriam generalizados
para região.

## 3. O que a curadoria remove, e quanto?

In [ ]:
from medflow_ai.data.anonymization import contains_pii
from medflow_ai.fine_tuning.dataset import (
    DatasetStats, anonymize_examples, curate, generate_examples,
)

brutos = generate_examples()
print(f"Exemplos gerados: {len(brutos)}")

com_pii = [e for e in brutos if contains_pii(e.instruction) or contains_pii(e.output)]
print(f"Exemplos com PII antes da anonimização: {len(com_pii)}")

limpos, rel_anon = anonymize_examples(brutos)
print(f"Identificadores removidos: {rel_anon.counts_by_kind}")

stats = DatasetStats(gerados=len(limpos))
curados = curate(limpos, stats)
print("\nEstatísticas de curadoria:")
print(json.dumps(stats.to_dict(), ensure_ascii=False, indent=2))

In [ ]:
import collections

familias = collections.Counter(e.familia for e in curados)
largura = max(familias.values())
for familia, total in familias.most_common():
    print(f"{familia:20s} {total:3d} {'█' * int(40 * total / largura)}")

**Interpretação.** A curadoria aplica seis filtros (duplicidade, comprimento mínimo e máximo,
idioma, PII residual). Nenhum exemplo é descartado por PII **depois** da anonimização — que é o
resultado desejado: o filtro existe como rede de segurança, não como mecanismo principal.

**Observação honesta.** O volume final é pequeno para fine-tuning. É suficiente para demonstrar
adaptação de **formato e comportamento**, e insuficiente para ensinar conhecimento clínico novo — o que
é exatamente a divisão de responsabilidades adotada (conhecimento fica no RAG).

## 4. Como o split evita data leakage?

Regra: **split por documento**, não por pergunta. Três documentos inteiros são reservados ao teste. Se
dividíssemos perguntas aleatoriamente, uma pergunta de treino e outra de teste poderiam vir da mesma
seção, tornando o benchmark trivial.

In [ ]:
from medflow_ai.fine_tuning.dataset import split_by_document

splits = split_by_document(curados)
for nome, itens in splits.items():
    docs = sorted({e.doc_id for e in itens})
    print(f"{nome:12s} {len(itens):3d} exemplos | {len(docs)} documentos")

treino = {e.doc_id for e in splits["train"]} | {e.doc_id for e in splits["validation"]}
teste = {e.doc_id for e in splits["test"]}
print(f"\nDocumentos reservados ao teste: {sorted(teste)}")
print(f"Interseção treino ∩ teste: {sorted(treino & teste) or 'VAZIA ✅'}")

In [ ]:
from medflow_ai.fine_tuning.dataset import build_sft_dataset

splits, stats, caminhos = build_sft_dataset()
print("Arquivos gerados:")
for nome, caminho in caminhos.items():
    print(f"  {nome:12s} {caminho}")

manifesto = json.loads(caminhos["manifest"].read_text(encoding="utf-8"))
print("\nManifesto (trecho):")
print(json.dumps({k: manifesto[k] for k in ("seed", "held_out_documents", "splits")},
                 ensure_ascii=False, indent=2))

In [ ]:
# O dataset NÃO depende do estado do banco: reconstruir com outro tamanho não muda nada.
build_synthetic_database(n_patients=8, seed=123)
_, stats_8, _ = build_sft_dataset(write=False)
build_synthetic_database(n_patients=40)
_, stats_40, _ = build_sft_dataset(write=False)

print(f"com 8 pacientes no banco : {stats_8.finais} exemplos | {stats_8.por_familia}")
print(f"com 40 pacientes no banco: {stats_40.finais} exemplos | {stats_40.por_familia}")
assert stats_8.to_dict() == stats_40.to_dict(), "o dataset não pode depender do banco"
print("\nInvariância confirmada ✅")

In [ ]:
# Exemplo final, no formato de chat consumido pelo SFTTrainer
exemplo = splits["train"][0].to_chat()
for mensagem in exemplo["messages"]:
    print(f"--- {mensagem['role'].upper()} ---")
    print(mensagem["content"][:600])
    print()

**Interpretação.** A interseção entre documentos de treino e de teste é vazia por construção, e o
manifesto registra seed, contagens e *fingerprints* de cada exemplo — qualquer pessoa pode verificar que
o split não mudou entre uma execução e outra.

**Limitação declarada.** O split por documento controla o leakage do *fine-tuning*. Ele não elimina a
limitação inerente da avaliação de RAG, em que o mesmo corpus é indexado e consultado; isso é medido e
declarado separadamente no notebook 03.

## 5. Persistência no Google Drive

In [ ]:
# Relatório de anonimização e estatísticas de curadoria como artefatos citáveis
relatorio_execucao = {
    "corpus": {"documentos": len(documentos), "secoes": sum(len(d.sections) for d in documentos)},
    "anonimizacao": rel_anon.to_dict(),
    "curadoria": stats.to_dict(),
    "splits": {nome: len(itens) for nome, itens in splits.items()},
    "held_out_documents": manifesto["held_out_documents"],
    "invariancia_ao_banco": stats_8.to_dict() == stats_40.to_dict(),
}
saida_local = PASTAS["artifacts"] / "01_preprocessing_report.json"
saida_local.write_text(json.dumps(relatorio_execucao, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(relatorio_execucao["curadoria"]["por_familia"], ensure_ascii=False, indent=2))
print(f"\nRelatório salvo em {saida_local}")

In [ ]:
# Persistência no Google Drive — uma execução só termina quando os resultados
# saem de /content. Fora do Colab, os mesmos arquivos vão para artifacts/colab/.
from medflow_ai.colab import persist, summarize

_relatorios = [
    persist([saida_local], PASTAS["artifacts"]),
    persist(
        [caminhos["train"], caminhos["validation"], caminhos["test"], caminhos["manifest"]],
        PASTAS["datasets"],
    ),
    persist([caminhos["manifest"]], PASTAS["shared/manifests"]),
]

print(summarize(_relatorios, titulo="RESUMO DA PERSISTÊNCIA — 01_preprocessing"))


## 6. Conclusão

| Pergunta | Resposta |
|---|---|
| A anonimização funciona? | Sim: 8 classes de identificador removidas, conteúdo clínico preservado |
| A curadoria é mensurável? | Sim: estatísticas antes/depois versionadas no manifesto |
| Há controle de leakage? | Sim: split por documento, com interseção vazia verificada |
| O dataset depende do banco? | Não: verificado com 8 e com 40 pacientes |
| O dataset é suficiente? | Para formato/comportamento sim; para conhecimento clínico não (papel do RAG) |

Próximo notebook: **`02_fine_tuning_qlora.ipynb`** — requer GPU.